In [2]:
import numpy as np
import pandas as pd

# Minimum Image Convention

In a system with periodic boundary conditions, like a simulation box in molecular dynamics in physics, the space "wraps around".

Suppose we want to compute the displacement vector between two points $x_1 = 0.2$ and $x_2 = 0.9$. Naively: $\Delta x = 0.7$. But this is not the shortest path, it's only 0.3 if we go backwards.

Minimum Image Convention redefines the displacement to be the shortest path on the periodic domain, ensuring $Δx∈[−L/2,+L/2].$

In [3]:
def minimum_image(dx: float, box_length: float) -> float:
    """Returns the minimum image of a coordinate difference."""
    reciprocal_half_box = 1.0 / (0.5 * box_length)  # Multiply Δx by this value to get how many "half-boxes" it spans.
    k = int(dx * reciprocal_half_box)  # k will only be 0, 1 for a 1D box.
    return dx - k * box_length  # Subtract k⋅L to pull it into the interval [−L/2,L/2]

In [6]:
for n in range(5):
    print(f"n = {n}, minimum_image({n}, 5) = {minimum_image(n, 5)}")

n = 0, minimum_image(0, 5) = 0
n = 1, minimum_image(1, 5) = 1
n = 2, minimum_image(2, 5) = 2
n = 3, minimum_image(3, 5) = -2
n = 4, minimum_image(4, 5) = -1


Once we know the shortest displacement Δ𝑥, we want to go halfway along that path starting from 𝑝1.

So we compute: p_mid = p1 + Δ𝑥/2.

But due to periodicity, this result might fall outside the domain $[0, L)$. Hence the `% box_length` ensures it wraps back into the primary box.


In [4]:
def midpoint_pbc(p1: float, p2: float, box_length: float) -> float:
    """Returns the midpoint between two positions under periodic boundary conditions."""
    dx = p2 - p1
    dx_mic = minimum_image(dx, box_length)
    midpoint = (p1 + 0.5 * dx_mic) % box_length
    return midpoint

In [5]:
print(minimum_image(0.7, 1))

-0.30000000000000004


In [9]:
box_length = 1.0
p1 = 0.2
p2 = 0.9

mid = midpoint_pbc(p1, p2, box_length)
print("Midpoint under PBC:", mid)  # Expected: 0.05

Midpoint under PBC: 0.04999999999999999


## Centroid under PBC:  Mapping to the unit circle

We can think of each coordinate as lying on a circle of circumference L and represent each coordinate value as an angle:
$$
\theta = \frac{2\pi x}{L}
$$
Then:
1. Convert each coordinate to $(\cos{\theta},\sin{\theta})$
2. Average the cosines and sines over all particles
3. Get the average angle via $\text{atan2} (\overline{\sin{\theta}}, \overline{\cos{\theta}})$
4. Map that angle back to $[0,L)$ to get the PBC-aware centroid coordinate.

In [49]:
def centroid_pbc(positions, box_length):
    """
    Computes the centroid of n particles in 3D under periodic boundary conditions.
    
    Parameters:
    - positions: numpy array of shape (n, 3), coordinates of particles
    - box_length: float, box size (assumes cubic box)
    
    Returns:
    - centroid: numpy array of shape (3,), the PBC-aware centroid
    """
    centroid = np.zeros(3)
    for dim in range(3):  # process x, y, z separately
        theta = 2 * np.pi * positions[:, dim] / box_length
        sin_sum = np.sum(np.sin(theta))
        cos_sum = np.sum(np.cos(theta))
        avg_theta = np.arctan2(sin_sum, cos_sum)
        if avg_theta < 0:  # ensure angle in [0, 2π)
            avg_theta += 2 * np.pi
        centroid[dim] = (avg_theta / (2 * np.pi)) * box_length
    return centroid

In [50]:
box_length = 1.0
positions = np.array([
    [0.2, 0.3, 0.4],
    [0.9, 0.8, 0.2],
    [0.1, 0.9, 0.7]
])

centroid = centroid_pbc(positions, box_length)
print("Centroid under PBC:", centroid)

Centroid under PBC: [0.07296583 0.9        0.4       ]
